<a href="https://colab.research.google.com/github/jmompou/03-MIAR-Algoritmos-de-Optimizacion/blob/main/JorgeIbanezPuertasTrabajo_Pr%C3%A1ctico_Algoritmos(V2%2Cno_borrar).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Jorge Ibáñez Puertas  <br>
Url: https://github.com/jmompou/03-MIAR-Algoritmos-de-Optimizacion<br>
Google Colab: https://colab.research.google.com/drive/1KV60A3EvBaYe2ssjLZ2eNy24oOUZLH1B?usp=sharing <br>

Problema:
>1. Sesiones de doblaje <br>

Descripción del problema:(copiar enunciado)

Problema 1. Organizar sesiones de doblaje(I)
Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:

Número de actores: 10

Número de tomas: 30

Actores/Tomas dataset: https://bit.ly/36D8IuK

1 indica que el actor participa en la toma

0 en caso contrario







                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

**Respuesta Modelo**
- **¿Como represento el espacio de soluciones?**
Un vector $X = (x_1, x_2, ..., x_{30})$ donde cada $x_i \in \{1, ..., D\}$ representa el día asignado a la toma $i$. El número máximo de días $D$ podría ser 30, aunque para minimizar costos será cercano a $ 30//6 = 5$. Cada solución válida debe cumplir la restricción de que ningún día tenga más de 6 tomas: $|\{i : x_i = d\}| \le 6 \quad
∀d$.
Alternativamente, el espacio de soluciones es el conjunto de todas las particiones válidas de las 30 tomas en subconjuntos de tamaño máximo 6.

- **¿Cual es la función objetivo?**
La función a minimizar es el gasto total en actores de doblaje. Como todos cobran lo mismo, se minimiza la suma del número de actores requeridos por día.
Sean $T_d$ las tomas programadas en el día $d$, y $A(t)$ los actores de la toma $t$. El conjunto de actores el día $d$ es $A_d = \bigcup_{t \in T_d} A(t)$.
Función objetivo: $\min Z = \sum_{d=1}^D |A_d|$.

- **¿Como implemento las restricciones?**
La restricción principal determina que un día no puede tener más de 6 tomas. Si usamos un algoritmo heurístico de construcción, simplemente verificamos que la capacidad del día no exceda 6 al añadir tomas. Si usamos búsqueda local o algoritmos genéticos, sólo se generan y exploran estados (mutaciones/cruces) que respeten la condición de tamaño $\le 6$ particiones, o se penalizan fuertemente las soluciones inviables en la función de evaluación.


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

**Respuesta Análisis**
- **¿Que complejidad tiene el problema? Orden de complejidad y Contabilizar el espacio de soluciones**
El problema es una generalización del problema de partición de conjuntos con costos dependientes de la unión de elementos (actores en este caso), con restricción de capacidad. Pertenece a la clase de problemas NP-Hard. No existe un algoritmo exacto polinomial para resolverlo, por lo que su orden de complejidad para resolución exacta asintóticamente es exponencial $O(c^n)$.
El número de formas de particionar 30 elementos en 5 subconjuntos (de exactamente 6 elementos) es:
$\frac{30!}{(6!)^5 \cdot 5!} \approx 1.37 \times 10^{14}$
Si consideramos configuraciones donde se usen más días (o días no balanceados, ej. 5, 5, 5, 5, 5, 5 no es lo único, podría ser 6, 6, 6, 6, 5, 1, etc.), el espacio es aún mayor. Explorar exhaustivamente este espacio mediante fuerza bruta resulta completamente inviable para los tiempos computacionales estándar.


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

**Respuesta Diseño**
- **¿Que técnica utilizo? ¿Por qué?**
Debido a la inmensidad del espacio de soluciones y su complejidad NP-Hard, se empleará un método aproximado (Heurística o Metaheurística).
A continuación propondremos un enfoque de dos fases mediante Algoritmo Voraz (Greedy) combinado con Búsqueda Local (Local Search):
1. **Algoritmo Greedy Construcitvo**: Ordena las tomas y va llenando los días intentando elegir tomas que minimicen la adición de nuevos actores al día actual. Si un día alcanza 6 tomas, se abre uno nuevo. Esto nos dará una solución inicial rápida y de buena calidad.
2. **Búsqueda Local**: Partiendo de la solución inicial, en cada iteración tomamos dos tomas de días distintos e intentamos intercambiarlas (operador SWAP) o movemos una toma a un día con espacio disponible (operador MOVE). Si la nueva configuración reduce el coste de actores, se acepta. Iteramos hasta alcanzar un óptimo local.
Esta aproximación se escoge por tener un excelente balance entre dificultad de implementación, velocidad de ejecución computacional y calidad en la minimización de la función de costos.


In [ ]:
import pandas as pd
import numpy as np
import random
import copy

# Cargar los datos desde el archivo Excel
# Como esto corre en el notebook o en local de forma aislada:
df = pd.read_excel('Datos problema doblaje(30 tomas, 10 actores).xlsx')
df = df.dropna(how='all') # Limpiar filas vacías

# Extraer la matriz de tomas
# Filtramos las filas que correspondan a Tomas (1 a 30)
tomas_df = df.iloc[1:31, 1:11]
tomas_df.index = df.iloc[1:31, 0].astype(int) # ID de Toma
tomas_df.columns = [f"Actor_{i+1}" for i in range(10)]
matriz_tomas = tomas_df.astype(int).values # array de 30x10

# Función para costo de un día
def costo_dia(tomas_asignadas):
    if len(tomas_asignadas) == 0:
        return 0
    # union of actors for these takes
    actores = np.sum(matriz_tomas[tomas_asignadas], axis=0)
    return np.count_nonzero(actores)

def costo_total(solucion):
    costo = 0
    for dia in solucion:
        costo += costo_dia(dia)
    return costo

# 1. Función Greedy Constructiva
def greedy_initial_solution():
    solucion = []
    tomas_pendientes = list(range(30))

    while tomas_pendientes:
        dia_actual = []
        actores_dia = np.zeros(10, dtype=int)

        # Seleccionar la primera toma (podríamos elegir la de mayores actores, pero el orden o una al azar vale)
        # Tomamos la que tiene más actores como semilla del día
        mejor_toma_inicial = max(tomas_pendientes, key=lambda t: np.sum(matriz_tomas[t]))
        dia_actual.append(mejor_toma_inicial)
        tomas_pendientes.remove(mejor_toma_inicial)
        actores_dia = matriz_tomas[mejor_toma_inicial]

        # Rellenar hasta 6
        while len(dia_actual) < 6 and tomas_pendientes:
            mejor_aumento = float('inf')
            mejor_toma = -1

            for t in tomas_pendientes:
                # Cuantos actores nuevos añade esta toma
                nuevos_actores = np.count_nonzero((matriz_tomas[t] > 0) & (actores_dia == 0))
                if nuevos_actores < mejor_aumento:
                    mejor_aumento = nuevos_actores
                    mejor_toma = t

            dia_actual.append(mejor_toma)
            tomas_pendientes.remove(mejor_toma)
            actores_dia = actores_dia | matriz_tomas[mejor_toma]

        solucion.append(dia_actual)
    return solucion

# 2. Búsqueda Local
def busqueda_local(solucion_inicial, max_iter=1000):
    mejor_solucion = copy.deepcopy(solucion_inicial)
    mejor_costo = costo_total(mejor_solucion)

    mejora = True
    iteracion = 0
    while mejora and iteracion < max_iter:
        mejora = False
        vecinos = []

        # Generar vecindario mediante SWAP (intercambio de una toma entre dos días)
        # y MOVE (mover una toma de un día a otro que tenga menos de 6)

        for d1 in range(len(mejor_solucion)):
            for d2 in range(len(mejor_solucion)):
                if d1 >= d2: continue

                # Intentar SWAP
                for i1, t1 in enumerate(mejor_solucion[d1]):
                    for i2, t2 in enumerate(mejor_solucion[d2]):
                        vecino = copy.deepcopy(mejor_solucion)
                        vecino[d1][i1] = t2
                        vecino[d2][i2] = t1
                        if costo_total(vecino) < mejor_costo:
                            mejor_solucion = vecino
                            mejor_costo = costo_total(vecino)
                            mejora = True

                # Intentar MOVE de d1 a d2
                if len(mejor_solucion[d2]) < 6:
                    for i1, t1 in enumerate(mejor_solucion[d1]):
                        vecino = copy.deepcopy(mejor_solucion)
                        t = vecino[d1].pop(i1)
                        vecino[d2].append(t)
                        # Eliminar el día si quedó vacío para no iterar de más, aunque afectaría índices
                        if len(vecino[d1]) == 0:
                            vecino.pop(d1)
                        if costo_total(vecino) < mejor_costo:
                            mejor_solucion = vecino
                            mejor_costo = costo_total(vecino)
                            mejora = True
                            break

                # Intentar MOVE de d2 a d1
                if len(mejor_solucion[d1]) < 6:
                    for i2, t2 in enumerate(mejor_solucion[d2]):
                        vecino = copy.deepcopy(mejor_solucion)
                        t = vecino[d2].pop(i2)
                        vecino[d1].append(t)
                        if len(vecino[d2]) == 0:
                            vecino.pop(d2)
                        if costo_total(vecino) < mejor_costo:
                            mejor_solucion = vecino
                            mejor_costo = costo_total(vecino)
                            mejora = True
                            break

        iteracion += 1

    return mejor_solucion, mejor_costo

solucion_greedy = greedy_initial_solution()
costo_greedy = costo_total(solucion_greedy)
print(f"Costo con Algoritmo Greedy Inicial: {costo_greedy} pagos a actores (Dias usados: {len(solucion_greedy)})")

solucion_final, costo_final = busqueda_local(solucion_greedy)
print(f"Costo final tras Búsqueda Local: {costo_final} pagos a actores (Dias usados: {len(solucion_final)})")

print("\n--- Planificación Días (Tomas 1-indexadas igual que en el Excel) ---")
for i, dia in enumerate(solucion_final):
    # Sumar 1 porque los índices en mi matriz_tomas son 0-indizados y corresponden a las tomas 1 a 30
    tomas_reales = [t+1 for t in dia]
    print(f"Día {i+1}: Tomas {tomas_reales} | Costo (actores requeridos): {costo_dia(dia)}")


Costo con Algoritmo Greedy Inicial: 31 pagos a actores (Dias usados: 5)
Costo final tras Búsqueda Local: 31 pagos a actores (Dias usados: 5)

--- Planificación Días (Tomas 1-indexadas igual que en el Excel) ---
Día 1: Tomas [1, 2, 6, 7, 9, 13] | Costo (actores requeridos): 5
Día 2: Tomas [11, 17, 19, 23, 3, 4] | Costo (actores requeridos): 6
Día 3: Tomas [12, 8, 14, 18, 22, 24] | Costo (actores requeridos): 5
Día 4: Tomas [10, 15, 21, 5, 28, 30] | Costo (actores requeridos): 7
Día 5: Tomas [20, 27, 16, 25, 26, 29] | Costo (actores requeridos): 8
